In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("/home/chupchik/Рабочий стол/fisrt_stepD/The_final_recomendation/output_dataset/ait_ads_wazuh_raw.parquet")

df = pd.read_parquet(DATA_PATH)

print(df.shape)
df.head()

(2493428, 15)


,scenario,timestamp,agent_id,agent_name,agent_ip,hostname,program,location,full_log,rule_id,rule_level,rule_description,rule_groups,rule_groups_str,y_priority
0,fox,2022-01-15 02:32:32+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
1,fox,2022-01-15 02:32:32+00:00,6,wazuh-client,192.168.128.170,taylorcruz-mail,freshclam,/var/log/syslog,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
2,fox,2022-01-15 02:32:37+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
3,fox,2022-01-15 02:32:42+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:42 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
4,fox,2022-01-15 02:32:47+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:47 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2493428 entries, 0 to 2493427
Data columns (total 15 columns):
 #   Column            Dtype              
---  ------            -----              
 0   scenario          str                
 1   timestamp         datetime64[us, UTC]
 2   agent_id          str                
 3   agent_name        str                
 4   agent_ip          str                
 5   hostname          str                
 6   program           str                
 7   location          str                
 8   full_log          str                
 9   rule_id           str                
 10  rule_level        int64              
 11  rule_description  str                
 12  rule_groups       object             
 13  rule_groups_str   str                
 14  y_priority        str                
dtypes: datetime64[us, UTC](1), int64(1), object(1), str(12)
memory usage: 995.1+ MB


In [3]:
print(df["y_priority"].value_counts())
print()
print(df["y_priority"].value_counts(normalize=True).round(4))

y_priority
medium      1873575
low          486285
high         131901
critical       1667
Name: count, dtype: int64

y_priority
medium      0.7514
low         0.1950
high        0.0529
critical    0.0007
Name: proportion, dtype: float64


In [4]:
missing = df.isna().mean().sort_values(ascending=False)
missing


full_log            0.080668
timestamp           0.000000
scenario            0.000000
agent_name          0.000000
agent_ip            0.000000
hostname            0.000000
agent_id            0.000000
program             0.000000
location            0.000000
rule_id             0.000000
rule_level          0.000000
rule_description    0.000000
rule_groups         0.000000
rule_groups_str     0.000000
y_priority          0.000000
dtype: float64

In [5]:
cols_to_check = [
    "scenario", "agent_name", "program", "location",
    "rule_id", "rule_description", "y_priority"
]

for col in cols_to_check:
    print(f"\n===== {col} =====")
    print("unique:", df[col].nunique(dropna=False))
    print(df[col].value_counts(dropna=False).head(10))


===== scenario =====
unique: 8
scenario
wilson             592771
wheeler            579133
harrison           573732
fox                457092
santos             117452
wardbeck            76022
shaw                60271
russellmitchell     36955
Name: count, dtype: int64

===== agent_name =====
unique: 1
agent_name
wazuh-client    2493428
Name: count, dtype: int64

===== program =====
unique: 13
program
web-accesslog      1696178
snort               306327
dovecot             277989
json                201141
freshclam             7475
apache-errorlog       2691
auth                   727
HORDE                  426
sshd                   181
auditd                 112
Name: count, dtype: int64

===== location =====
unique: 15
location
/var/log/apache2/intranet-access.log        1695733
/var/log/suricata/fast.log                   306327
/var/log/suricata/eve.json                   201141
/var/log/syslog                              100351
/var/log/mail.info                          

In [6]:
rule_to_priority = df.groupby("rule_id")["y_priority"].nunique()

print("rule_id с 1 уникальным классом:", (rule_to_priority == 1).sum())
print("всего rule_id:", len(rule_to_priority))
print("доля:", ((rule_to_priority == 1).sum() / len(rule_to_priority)).round(4))

rule_id с 1 уникальным классом: 31
всего rule_id: 31
доля: 1.0


In [7]:
df.groupby(["rule_id", "y_priority"]).size().reset_index(name="count").sort_values(
    ["rule_id", "count"], ascending=[True, False]
).head(20)

,rule_id,y_priority,count
0,20100,high,626
1,20101,medium,294636
2,20151,high,2362
3,20152,high,7036
4,20161,critical,202
5,20162,critical,1465
6,2501,medium,426
7,30305,medium,2353
8,30306,medium,338
9,31101,medium,1571311


In [8]:
print(df["rule_level"].describe())
print()
print(df["rule_level"].value_counts().sort_index().head(20))

count    2.493428e+06
mean     4.997173e+00
std      1.497281e+00
min      3.000000e+00
25%      5.000000e+00
50%      5.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: rule_level, dtype: float64

rule_level
3      486285
4          13
5     1576276
6      297286
8         629
10     131272
11       1667
Name: count, dtype: int64


In [9]:
print(df["scenario"].value_counts())

scenario
wilson             592771
wheeler            579133
harrison           573732
fox                457092
santos             117452
wardbeck            76022
shaw                60271
russellmitchell     36955
Name: count, dtype: int64
